# 10 · Quarantine Bad Records

**Layer:** Bronze &nbsp;|&nbsp; **Stage:** exception handling &nbsp;|&nbsp; **Target:** `banking_bronze.quarantine_records`

The **failure branch** of the pipeline. `bronze_quality_gate` routes here when
`03_validate_bronze` returns `FAIL`, instead of promoting anything to silver.

Rather than aborting the run, this notebook isolates exactly which records
broke which rule, writes them to a quarantine table with a machine-readable
reason and a suggested remediation, and produces a triage summary that a data
steward can work from. The pipeline then continues to `09_publish_and_audit`
(`run_if: All Done`), so a failed run still leaves a complete audit trail.


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.utils import AnalysisException

spark = SparkSession.builder.appName("medallion-quarantine").enableHiveSupport().getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "16")

RUN_DATE  = "2026-09-03"
BATCH_ID  = f"batch_{RUN_DATE.replace('-', '')}"
BRONZE_DB = "banking_bronze"

VALID_CURRENCIES = ["INR", "USD", "EUR", "GBP", "AED", "SGD"]

accounts = spark.table(f"{BRONZE_DB}.raw_accounts")
txns     = spark.table(f"{BRONZE_DB}.raw_transactions")

In [ ]:
REMEDIATION = {
    "NULL_PRIMARY_KEY":     "reject at source; require non-null key in the landing contract",
    "DUPLICATE_PRIMARY_KEY": "safe to auto-dedupe; confirm the switch replay window",
    "INVALID_CURRENCY":     "map to ISO-4217 or add the code to the whitelist",
    "NON_POSITIVE_AMOUNT":  "auth holds and reversals must land on the adjustments feed",
    "UNKNOWN_ACCOUNT":      "await the next account master delta, then reprocess",
    "MISSING_CUSTOMER":     "backfill from CRM; blocks customer-level reporting",
}

def quarantine(df, entity, reason, key_col):
    """Normalise any offending subset into the common quarantine shape."""
    return (
        df.select(
            F.col(key_col).cast("string").alias("record_key"),
            F.lit(entity).alias("source_entity"),
            F.lit(reason).alias("reject_reason"),
            F.lit(REMEDIATION[reason]).alias("remediation"),
            F.to_json(F.struct([df[c] for c in df.columns if not c.startswith("_")])).alias("record_payload"),
        )
    )

In [ ]:
buckets = []

# --- account defects ---------------------------------------------------
buckets.append(quarantine(
    accounts.filter(F.col("account_id").isNull()),
    "raw_accounts", "NULL_PRIMARY_KEY", "customer_id"))

dupe_accounts = (
    accounts.filter(F.col("account_id").isNotNull())
    .groupBy("account_id").count().filter(F.col("count") > 1).select("account_id")
)
buckets.append(quarantine(
    accounts.join(dupe_accounts, on="account_id", how="left_semi"),
    "raw_accounts", "DUPLICATE_PRIMARY_KEY", "account_id"))

buckets.append(quarantine(
    accounts.filter(F.col("customer_id").isNull() & F.col("account_id").isNotNull()),
    "raw_accounts", "MISSING_CUSTOMER", "account_id"))

In [ ]:
# --- transaction defects ------------------------------------------------
buckets.append(quarantine(
    txns.filter(F.col("txn_id").isNull()),
    "raw_transactions", "NULL_PRIMARY_KEY", "account_id"))

dupe_txns = (
    txns.filter(F.col("txn_id").isNotNull())
    .groupBy("txn_id").count().filter(F.col("count") > 1).select("txn_id")
)
buckets.append(quarantine(
    txns.join(dupe_txns, on="txn_id", how="left_semi"),
    "raw_transactions", "DUPLICATE_PRIMARY_KEY", "txn_id"))

buckets.append(quarantine(
    txns.filter(~F.upper(F.col("currency")).isin(VALID_CURRENCIES)),
    "raw_transactions", "INVALID_CURRENCY", "txn_id"))

buckets.append(quarantine(
    txns.filter(F.col("amount") <= 0),
    "raw_transactions", "NON_POSITIVE_AMOUNT", "txn_id"))

known = accounts.filter(F.col("account_id").isNotNull()).select("account_id").distinct()
buckets.append(quarantine(
    txns.join(known, on="account_id", how="left_anti"),
    "raw_transactions", "UNKNOWN_ACCOUNT", "txn_id"))

In [ ]:
from functools import reduce

quarantined = (
    reduce(lambda a, b: a.unionByName(b), buckets)
    .withColumn("_batch_id", F.lit(BATCH_ID))
    .withColumn("_run_date", F.lit(RUN_DATE))
    .withColumn("_quarantined_at", F.current_timestamp())
)

(
    quarantined.write.mode("overwrite").format("parquet")
    .saveAsTable(f"{BRONZE_DB}.quarantine_records")
)

out = spark.table(f"{BRONZE_DB}.quarantine_records")
print(f"quarantined {out.count()} record(s) to {BRONZE_DB}.quarantine_records")

In [ ]:
print("triage summary")

out.groupBy("source_entity", "reject_reason", "remediation").count() \
   .orderBy(F.desc("count")).show(truncate=False)

In [ ]:
print("quarantined records")

out.select("source_entity", "record_key", "reject_reason", "record_payload") \
   .orderBy("source_entity", "reject_reason").show(50, truncate=False)

print("=" * 60)
print(f"QUARANTINE_COUNT={out.count()}")
print("Bronze was not promoted to silver for this batch.")
print("=" * 60)